# Introduction

## Overview

## Purpose

# Data Acquisition

## Downloading data (gdown)

In [1]:
# getting the data from google drive (can run for a little long, data has 2,4GB)
!gdown --quiet 104WLoNAy02-r7d18ZE16vudY0puVpxm2

## Loading relevant packages and setting logging

In [2]:
import sys
import logging
from pathlib import Path    # creating dir

import pandas as pd         # data management

In [3]:
# Configure logging
log_filename = "../logs/pre_processing.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(log_filename, mode="w"),  # Overwrite log file
        logging.StreamHandler(sys.stdout)  # Print to console
    ]
)

## Utility Functions

In [4]:
# formatting to save files 
def formatting_label_name(str: str) -> str:
  return str.title().replace(" ","")

In [5]:
size_limit = 10000 # maximum amount of rows for datasets, since some labels have many instances
rnd_state = 42 # ensures the same sample for the same data

# if a label has more than num_rows instances, it returns a random sample of `num_rows` instances
def create_datasets(data: pd.DataFrame, label: str, num_rows: int = size_limit, random_state: int = 42) -> pd.DataFrame:
    dataset = data[data['predict'] == label]
    return dataset.sample(num_rows, random_state=random_state) if len(dataset) > num_rows else dataset

# Data Exploration

## Loading the Dataset into a Pandas DataFrame

In [6]:
df_event_data = pd.read_pickle("df_event_data_with_embeddings.pkl") 

In [7]:
df_event_data #run if you want to visualize the dataframe

,gid,url,geoinfo1,taxon1,taxon2,geoinfo2,who1,who2,text,predict,predict_proba,country,country_code,country_lat,country_lng,embeddings
924124,20200214223000-56,https://www.hppr.org/post/how-covid-19-kills-n...,,KILL;CRISISLEX_T03_DEAD;TAX_DISEASE;TAX_DISEAS...,"CRISISLEX_CRISISLEXREC,3366;TAX_DISEASE_INFECT...","4#Wuhan, Hubei, China#CH#CH12#13124#30.5833#11...","Anthony Fauci,3509;Yoko Furuya,1354;Sylvie Bri...","Columbia University Irving Medical Center,1434...",How COVID-19 Kills: The New Coronavirus Diseas...,covid-19 vaccine,1.000000,"Wuhan, Hubei, China",CH,30.5833,114.267,"[-0.020972056, -0.12172927, -0.007201001, -0.0..."
2284004,20200228053000-1137,http://www.ecns.cn/news/politics/2020-02-28/de...,KILL#13##1#South Korea#KS#KS#37#127.5#KS#2101;...,TAX_ETHNICITY;TAX_ETHNICITY_CHINESE;TAX_WORLDL...,"TAX_DISEASE_EPIDEMIC,1288;TAX_DISEASE_EPIDEMIC...",1#South Korea#KS#KS##37#127.5#KS#33;1#South Ko...,"Xing Haiming,46;Xing Haiming,240;Xing Haiming,...","Chinese Embassy,157;Chinese Embassy,351;Chines...",Chinese Embassy in South Korea donates masks t...,funding for covid,1.000000,South Korea,KS,37,127.5,"[0.05996564, -0.16531578, -0.0031238094, 0.057..."
1181064,20200224034500-218,https://www.rnz.co.nz/news/national/410233/cov...,,TAX_FNCACT;TAX_FNCACT_MINISTER;LEADER;TAX_FNCA...,"CRISISLEX_CRISISLEXREC,1466;UNGP_HEALTHCARE,74...",1#South Korea#KS#KS##37#127.5#KS#1508;1#Chines...,"Jacinda Arden,28","World Health Organisation,1182",Covid-19: Travel restrictions for those coming...,covid-19 restrictions lifted,1.000000,South Korea,KS,37,127.5,"[-0.104065694, 0.03418176, -0.0051460676, -0.0..."
3572684,20200224134500-9,https://www.chinanationalnews.com/news/2641216...,KILL#150##1#China#CH#CH#35#105#CH#1919;KILL#2#...,TAX_ETHNICITY;TAX_ETHNICITY_INDIAN;TAX_FNCACT;...,"GENERAL_HEALTH,1852;MEDICAL,1852;GENERAL_GOVER...",1#South Korea#KS#KS##37#127.5#KS#18;1#South Ko...,,"South China Morning Post,1835;Yonhap News Agen...","COVID-19: Reconsider travelling to S Korea, In...",hospital for covid-19 patients,0.333333,South Korea,KS,37,127.5,"[-0.043811183, -0.0037411905, -0.005606968, 0...."
3188350,20200224071500-424,https://www.thenorthernecho.co.uk/news/nationa...,KILL#2##1#China#CH#CH#35#105#CH#134;KILL#8##1#...,KILL;TAX_FNCACT;TAX_FNCACT_OFFICIALS;TAX_DISEA...,"GENERAL_GOVERNMENT,2051;EPU_POLICY_GOVERNMENT,...","4#Milan, Lombardia, Italy#IT#IT09#18363#45.466...",,"Xinhua News Agency,1731;Xinhua,1719;Xinhua,1798",South Korea sees increase in recorded Covid-19...,covid-19 cases emerge,0.666667,"Milan, Lombardia, Italy",IT,45.4667,9.2,"[-0.026586916, -0.3584028, -0.0059149084, -0.0..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1029937,20210309233000-1970,https://www.tbnewswatch.com/local-news/thunder...,,SECURITY_SERVICES;TAX_FNCACT;TAX_FNCACT_POLICE...,"MEDICAL,768;MEDICAL,1151;TAX_FNCACT_PARAMEDICS...","4#Thunder Bay, Ontario, Canada#CA#CA08#12684#4...","Bill Mauro,1354;Kristen Oliver,153",,Thunder Bay frontline officers receive first d...,covid-19 vaccine,1.000000,"Thunder Bay, Ontario, Canada",CA,48.4,-89.2333,"[-0.074297085, -0.2056211, -0.0043920088, -0.1..."
180173,20210308140000-411,https://www.ross-shirejournal.co.uk/news/ross-...,,TAX_FNCACT;TAX_FNCACT_WOMEN;EPU_ECONOMY;EPU_EC...,"GENERAL_GOVERNMENT,1355;EPU_POLICY_GOVERNMENT,...",1#Scotland#UK#UK##54#-4#UK#428;1#Scotland#UK#U...,"Maree Todd,24","Scottish Government Women Returners Programme,...",Women have suffered 'disproportionately' from ...,impact of the covid-19,1.000000,Scotland,UK,54,-4,"[-0.10385181, 0.0008548315, -0.0063411416, -0...."
703959,20210302051500-673,https://www.republicworld.com/world-news/pakis...,KILL#896##1#Pakistan#PK#PK#30#70#PK#647;,GENERAL_HEALTH;MEDICAL;TAX_DISEASE;TAX_DISEASE...,"GENERAL_GOVERNMENT,1807;GENERAL_GOVERNMENT,216...",1#Pakistani#PK#PK##30#70#PK#141;1#Pakistan#PK#...,"Qaisar Sajjad,1518","Pakistan National,689;Pakistan Medical Associa...",Pakistan's NCOC raises alarm over dull vaccine...,covid-19 vaccine,1.0

In [8]:
# getting the relevant columns (.copy to avoid a view)
df_selected_features = df_event_data[["gid","text","predict","country_lat","country_lng","embeddings"]].copy() 

In [9]:
df_selected_features

,gid,text,predict,country_lat,country_lng,embeddings
924124,20200214223000-56,How COVID-19 Kills: The New Coronavirus Diseas...,covid-19 vaccine,30.5833,114.267,"[-0.020972056, -0.12172927, -0.007201001, -0.0..."
2284004,20200228053000-1137,Chinese Embassy in South Korea donates masks t...,funding for covid,37,127.5,"[0.05996564, -0.16531578, -0.0031238094, 0.057..."
1181064,20200224034500-218,Covid-19: Travel restrictions for those coming...,covid-19 restrictions lifted,37,127.5,"[-0.104065694, 0.03418176, -0.0051460676, -0.0..."
3572684,20200224134500-9,"COVID-19: Reconsider travelling to S Korea, In...",hospital for covid-19 patients,37,127.5,"[-0.043811183, -0.0037411905, -0.005606968, 0...."
3188350,20200224071500-424,South Korea sees increase in recorded Covid-19...,covid-19 cases emerge,45.4667,9.2,"[-0.026586916, -0.3584028, -0.0059149084, -0.0..."
...,...,...,...,...,...,...
1029937,20210309233000-1970,Thunder Bay frontline officers receive first d...,covid-19 vaccine,48.4,-89.2333,"[-0.074297085, -0.2056211, -0.0043920088, -0.1..."
180173,20210308140000-411,Women have suffered 'disproportionately' from ...,impact of the covid-19,54,-4,"[-0.10385181, 0.0008548315, -0.0063411416, -0...."
703959,20210302051500-673,Pakistan's NCOC raises alarm over dull vaccine...,covid-19 vaccine,30,70,"[0.018137967, 0.01642545, -0.0037686664, -0.07..."
2108475,20210520121500-755,United Airlines adding flights worldwide as Co...,covid-19 restrictions eased,40.3363,-89.0022,"[-0.07282667, 0.09720857, -0.0031480393, -0.07..."


# Feature Selection & Data Cleaning

In [10]:
# verifying the data types for each column
df_selected_features.info()

<class 'pandas.core.frame.DataFrame'>
Index: 392105 entries, 924124 to 486543
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   gid          392105 non-null  object
 1   text         392105 non-null  object
 2   predict      392105 non-null  object
 3   country_lat  392105 non-null  object
 4   country_lng  392105 non-null  object
 5   embeddings   392105 non-null  object
dtypes: object(6)
memory usage: 20.9+ MB


In [11]:
# transforming the latitude and longitude in float type to allow oerations
df_selected_features['country_lat'] = pd.to_numeric(df_selected_features['country_lat'])
df_selected_features['country_lng'] = pd.to_numeric(df_selected_features['country_lng'])

In [12]:
# dropping data without geoespatial coordinates
df_selected_features = df_selected_features.dropna(subset={'country_lat','country_lng'})

In [13]:
# getting the dates from the gID 
dates = [date[:4]+'-'+date[4:6]+'-'+date[6:8] for date in df_selected_features.gid]

df_selected_features.loc[:, 'dates'] = pd.to_datetime(dates, errors='coerce')

/tmp/ipykernel_741427/1960585449.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected_features.loc[:, 'dates'] = pd.to_datetime(dates, errors='coerce')


In [14]:
df_selected_features.info() # just to see wheter or not changes were successful

<class 'pandas.core.frame.DataFrame'>
Index: 336195 entries, 924124 to 486543
Data columns (total 7 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   gid          336195 non-null  object        
 1   text         336195 non-null  object        
 2   predict      336195 non-null  object        
 3   country_lat  336195 non-null  float64       
 4   country_lng  336195 non-null  float64       
 5   embeddings   336195 non-null  object        
 6   dates        336195 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(2), object(4)
memory usage: 20.5+ MB


In [15]:
df_selected_features

,gid,text,predict,country_lat,country_lng,embeddings,dates
924124,20200214223000-56,How COVID-19 Kills: The New Coronavirus Diseas...,covid-19 vaccine,30.5833,114.26700,"[-0.020972056, -0.12172927, -0.007201001, -0.0...",2020-02-14
2284004,20200228053000-1137,Chinese Embassy in South Korea donates masks t...,funding for covid,37.0000,127.50000,"[0.05996564, -0.16531578, -0.0031238094, 0.057...",2020-02-28
1181064,20200224034500-218,Covid-19: Travel restrictions for those coming...,covid-19 restrictions lifted,37.0000,127.50000,"[-0.104065694, 0.03418176, -0.0051460676, -0.0...",2020-02-24
3572684,20200224134500-9,"COVID-19: Reconsider travelling to S Korea, In...",hospital for covid-19 patients,37.0000,127.50000,"[-0.043811183, -0.0037411905, -0.005606968, 0....",2020-02-24
3188350,20200224071500-424,South Korea sees increase in recorded Covid-19...,covid-19 cases emerge,45.4667,9.20000,"[-0.026586916, -0.3584028, -0.0059149084, -0.0...",2020-02-24
...,...,...,...,...,...,...,...
1029937,20210309233000-1970,Thunder Bay frontline officers receive first d...,covid-19 vaccine,48.4000,-89.23330,"[-0.074297085, -0.2056211, -0.0043920088, -0.1...",2021-03-09
180173,20210308140000-411,Women have suffered 'disproportionately' from ...,impact of the covid-19,54.0000,-4.00000,"[-0.10385181, 0.0008548315, -0.0063411416, -0....",2021-03-08
703959,20210302051500-673,Pakistan's NCOC raises alarm over dull vaccine...,covid-19 vaccine,30.0000,70.00000,"[0.018137967, 0.01642545, -0.0037686664, -0.07...",2021-03-02
2108475,20210520121500-755,United Airlines adding flights worldwide as Co...,covid-19 restrictions eased,40.3363,-89.00220,"[-0.07282667, 0.09720857, -0.0031480393, -0.07...",2021-05-20


# Creating Final Datasets

## Sampling strategies

In [16]:
# analyzing the amount of occurences for each label
df_selected_features['predict'].value_counts()

predict
covid-19 vaccine                   69975
 warned of covid-19                60502
 new covid-19 cases                31455
 deaths from covid-19              29308
 hospital for covid-19 patients    27199
covid-19 cases emerge              26270
 funding for covid                 23751
 tests for covid-19                21432
 covid-19 restrictions lifted       9182
 drug against covid-19              7302
 impact of the covid-19             6684
 covid-19 restrictions eased        5287
 treatments for covid-19            4130
 response to covid-19               2672
 reopening schools                  2087
 resurgence of covid-19             1592
 market analysis                    1185
 enforcing covid-19 protocols       1123
 breach covid rules                 1078
 festival cancelled                  997
 wears face mask                     898
 social distancing                   557
 quarantine for infected             470
 donate blood plasma                 387
research

In [17]:
# selecting only the labels with more than 1000 occurrences as they are more significant to our purposes
label_counts = df_selected_features['predict'].value_counts()
predicts = df_selected_features['predict'].unique()

labels = [label for label in predicts if label_counts[label] > 1000]

In [18]:
len(labels)

19

## Saving pre-processed data

In [19]:
# creates a path for the datasets
output_dir = "./UsageDatasets/"
Path(output_dir).mkdir(parents=True, exist_ok=True)

In [20]:
for label in labels:
  dataset = create_datasets(df_selected_features, label, size_limit, rnd_state)
  print(dataset.shape)
  dataset.to_pickle(f"{output_dir}/{formatting_label_name(label)}_dataset.pkl")
  break

(10000, 7)


In [21]:
logging.info("Preprocessing completed successfully")

2025-03-02 00:26:26,847 - INFO - Preprocessing completed successfully


Another approach could be filter datasets without considering labels. Just getting a defined amount of instances of the dataset, regardless its label...